# NCAA March Madness Feature Engineering Pipeline (Men & Women)

This notebook implements the data processing and feature engineering pipeline for predicting NCAA tournament outcomes. It covers:
1. Data Loading (M & W)
2. Aggregating Regular Season Statistics
3. Calculating Efficiency Ratings
4. Processing Seeds and Rankings
5. Creating the Combined Feature Set

In [ ]:
import pandas as pd
import numpy as np
import os

DATA_PATH = './march-machine-learning-mania-2026/'

def load_data(gender):
    prefix = gender.upper()
    teams = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}Teams.csv'))
    seasons = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}Seasons.csv'))
    seeds = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}NCAATourneySeeds.csv'))
    regular_results = pd.read_csv(os.path.join(DATA_PATH, f'{prefix}RegularSeasonDetailedResults.csv'))
    return teams, seasons, seeds, regular_results

m_teams, m_seasons, m_seeds, m_regular_results = load_data('M')
w_teams, w_seasons, w_seeds, w_regular_results = load_data('W')
m_rankings = pd.read_csv(os.path.join(DATA_PATH, 'MMasseyOrdinals.csv'))

print("Data loaded successfully for both genders.")

## 1. Feature Engineering Functions

Generic functions to process stats for either gender.

In [ ]:
def calculate_team_stats(df):
    w_cols = ['WScore', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']
    l_cols = ['LScore', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']
    
    win_stats = df[['Season', 'WTeamID'] + w_cols].rename(columns={'WTeamID': 'TeamID'})
    win_stats.columns = [c.replace('W', '') if c != 'Season' else c for c in win_stats.columns]
    
    lose_stats = df[['Season', 'LTeamID'] + l_cols].rename(columns={'LTeamID': 'TeamID'})
    lose_stats.columns = [c.replace('L', '') if c != 'Season' else c for c in lose_stats.columns]
    
    combined_stats = pd.concat([win_stats, lose_stats])
    seasonal_stats = combined_stats.groupby(['Season', 'TeamID']).mean().reset_index()
    return seasonal_stats

def add_efficiency_metrics(df):
    df['Possessions'] = df['FGA'] + 0.475 * df['FTA'] - df['OR'] + df['TO']
    df['OffEff'] = (df['Score'] / df['Possessions']) * 100
    df['eFG'] = (df['FGM'] + 0.5 * df['FGM3']) / df['FGA']
    return df

## 2. Processing Men's Features

In [ ]:
m_stats = calculate_team_stats(m_regular_results)
m_stats = add_efficiency_metrics(m_stats)
m_seeds['SeedInt'] = m_seeds['Seed'].apply(lambda x: int(x[1:3]))

m_final_rankings = m_rankings[m_rankings['RankingDayNum'] == 133].copy()
m_consensus_rank = m_final_rankings.groupby(['Season', 'TeamID'])['OrdinalRank'].mean().reset_index().rename(columns={'OrdinalRank': 'AvgRank'})

m_features = m_stats.merge(m_seeds[['Season', 'TeamID', 'SeedInt']], on=['Season', 'TeamID'], how='left')
m_features = m_features.merge(m_consensus_rank, on=['Season', 'TeamID'], how='left')
m_features['SeedInt'] = m_features['SeedInt'].fillna(20)
m_features['AvgRank'] = m_features['AvgRank'].fillna(m_features['AvgRank'].max())
print("Men's features ready.")

## 3. Processing Women's Features

In [ ]:
w_stats = calculate_team_stats(w_regular_results)
w_stats = add_efficiency_metrics(w_stats)
w_seeds['SeedInt'] = w_seeds['Seed'].apply(lambda x: int(x[1:3]))

# Note: No Massey Ordinals for Women in the provided dataset, using Seed as primary rank proxy
w_features = w_stats.merge(w_seeds[['Season', 'TeamID', 'SeedInt']], on=['Season', 'TeamID'], how='left')
w_features['SeedInt'] = w_features['SeedInt'].fillna(20)
w_features['AvgRank'] = w_features['SeedInt'] * 10 # Rough proxy since Massey is missing

print("Women's features ready.")

## 4. Combining and Saving

In [ ]:
all_features = pd.concat([m_features, w_features], ignore_index=True)
all_features.to_csv('all_team_features.csv', index=False)
print(f"Combined feature set saved to all_team_features.csv with {len(all_features)} rows.")